# RBF on GTSQA, v2: the paper configuration ("Ours")

https://huggingface.co/datasets/Graphcore/GTSQA

Differences from v1 (`rbf-train.ipynb`):
- **Stop factor φ is trained** (its own BCE over the steps that can end a path; v1 had the head but no loss, so φ stayed random).
- **Fine-tuned BGE-M3 question–relation similarity** as an input of π and φ (`REL_PRIOR='bge_ft'`): cosine of the BGE-M3 + LoRA encoder fine-tuned on WebQSP (`adapter_v1/final`, the same encoder as on WebQSP/CWQ). Upload that folder as a Kaggle dataset and set `BGE_ADAPTER`.
- `Z_ENCODER`: `'gte'` (default, as in the paper: GTE-large for the question / relation vectors) or `'bge-m3'` (BGE-M3 base instead of GTE).
- Model selection on validation **recall@100 of the labelled triples** (as on WebQSP/CWQ), not ansR@50.
- Test keeps the **top 200** triples and reports ansR@50/100/200 and the recall of the answer-subgraph triples.

| Phase | Work | Cleanup |
|---|---|---|
| 1. Convert | HF dataset → small `.pkl` shards + manifest (resumable) | delete HF cache |
| 2. Embedding | question + unique relation vectors (`Z_ENCODER`), lexical score | unload encoder |
| 2b. BGE prior | fine-tuned BGE-M3 cosine per (question, relation) | unload encoder |
| 3. Train | π + φ jointly, checkpoint every epoch, resumable | free GPU/RAM |
| 4. Test | top-200, metrics, `test_preds.json` | |

> After the `pip install` cell, **Restart session**, then run from the **Config** cell.


In [ ]:
# Kaggle hiện chạy Python 3.13: các bản ghim cũ (pyarrow==16.1.0, numpy==1.26.4, pandas==2.2.2, tokenizers 0.19.x...)
# KHÔNG có wheel cho 3.13 -> không cài được. Vì vậy: KHÔNG ghim numpy/pandas/pyarrow, KHÔNG uninstall gì cả.
# Chỉ đảm bảo `transformers` nằm trong khoảng chạy được code remote của gte-large-en-v1.5.
# Cell này chạy TRƯỚC mọi import nên không cần restart kernel.
import sys, re, subprocess, importlib.metadata as _md

print('Python', sys.version.split()[0])


def _ver(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return None


def _mm(v):
    return tuple(int(x) for x in re.findall(r'\d+', v)[:2])


def _pip(*pkgs):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', '--no-input', '--prefer-binary',
           '--disable-pip-version-check', *pkgs]
    print('>>', ' '.join(cmd[3:]))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print('!! pip lỗi (bỏ qua, chạy tiếp):', (r.stderr or r.stdout)[-600:])
    return r.returncode == 0


tf = _ver('transformers')
if tf is None or not ((4, 41) <= _mm(tf) < (4, 57)):
    _pip('transformers==4.51.3')            # pure-python + tokenizers 0.21 (abi3) -> có wheel cho 3.13
if _ver('sentence-transformers') is None:
    _pip('sentence-transformers')
pa = _ver('pyarrow')
if pa is None or _mm(pa) < (15, 0):
    _pip('pyarrow>=18')                      # 18.0.0 là bản đầu có wheel cho 3.13

for _p in ('transformers', 'sentence-transformers', 'safetensors', 'pyarrow', 'huggingface-hub', 'numpy', 'torch'):
    print(f'{_p:24s}', _ver(_p))

## Config + helper dọn dẹp / theo dõi tài nguyên

In [ ]:
import os, sys, json, pickle, re, math, time, random, gc, shutil, string, hashlib, copy
from collections import defaultdict
from functools import lru_cache
from pathlib import Path


# ---- Cache HF đưa ra thư mục tạm riêng (dễ xoá). Phải set TRƯỚC khi import datasets/transformers ----
def _pick_tmp():
    for p in ('/kaggle/temp', '/tmp'):
        try:
            os.makedirs(p, exist_ok=True)
            if os.access(p, os.W_OK):
                return p
        except Exception:
            pass
    return '/tmp'


TMP_DIR = _pick_tmp()
os.environ.setdefault('HF_HOME', f'{TMP_DIR}/hf')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # giảm phân mảnh VRAM
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
HF_HOME = Path(os.environ['HF_HOME'])
HF_DIRS = list({HF_HOME, Path.home() / '.cache' / 'huggingface'})

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

try:
    import psutil
except ImportError:
    psutil = None

# ---------------------------------------------------------------- run configuration (v2 = paper configuration)
Z_ENCODER = 'gte'                       # 'gte' (paper) | 'bge-m3' (BGE-M3 base vectors instead of GTE)
REL_PRIOR = 'bge_ft'                    # 'bge_ft' (paper: fine-tuned BGE-M3 cosine as an input of pi / phi) | 'none'
USE_STOP = True                         # train the stop factor phi (paper)
BGE_BASE = 'BAAI/bge-m3'
BGE_ADAPTER = '/kaggle/input/rbf-bge-adapter/final'   # <- Kaggle dataset with retrieve/rbf/data/bge/adapter_v1/final
RUN_NAME = f'gtsqa_rbf_v2_{Z_ENCODER}_{REL_PRIOR}' + ('_stop' if USE_STOP else '')
SELECT_METRIC = 'triple_recall@100'     # validation metric for model selection (as on WebQSP / CWQ)
TEST_TOPK = 200                         # triples kept per test question
# Output of the v1 notebook (shards + GTE embeddings), added as a Kaggle dataset; None = build everything from HF.
# Its embeddings are GTE, so they are reused only when Z_ENCODER == 'gte'.
REUSE_DIR = '/kaggle/input/gtsqa-rbf-processed/gtsqa'

EMB_MODEL = {'gte': 'Alibaba-NLP/gte-large-en-v1.5', 'bge-m3': 'BAAI/bge-m3'}[Z_ENCODER]
EMB_DIM = 1024                          # both encoders: 1024-d
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE_EMB = 128
BLOCK_SIZE = 2048                       # số câu hỏi thô / shard
OUT_DIR = Path('/kaggle/working/gtsqa')
OUT_DIR.mkdir(parents=True, exist_ok=True)


# ----------------------------------------------------------------------------- tiện ích
def dir_size_gb(p):
    p = Path(p)
    if not p.exists():
        return 0.0
    total = 0
    for root, _, files in os.walk(p):
        for f in files:
            try:
                total += os.path.getsize(os.path.join(root, f))
            except OSError:
                pass
    return total / 1e9


def report(tag=''):
    """In nhanh: đĩa còn trống, dung lượng OUT_DIR, cache HF, RAM, VRAM."""
    root = '/kaggle/working' if os.path.exists('/kaggle/working') else '/'
    du = shutil.disk_usage(root)
    msg = [f'[{tag}]' if tag else '[report]',
           f'disk free {du.free / 1e9:.1f}/{du.total / 1e9:.1f}GB',
           f'OUT_DIR {dir_size_gb(OUT_DIR):.2f}GB',
           f'HF cache {sum(dir_size_gb(d) for d in HF_DIRS):.2f}GB']
    if psutil:
        msg.append(f'RAM proc {psutil.Process().memory_info().rss / 1e9:.1f}GB, '
                   f'còn trống {psutil.virtual_memory().available / 1e9:.1f}GB')
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            msg.append(f'GPU{i} {torch.cuda.memory_allocated(i) / 1e9:.1f}GB used / '
                       f'{torch.cuda.memory_reserved(i) / 1e9:.1f}GB reserved')
    print(' | '.join(msg))


def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def clear_hf_cache(subdirs=('datasets/downloads', 'hub', 'datasets', 'modules')):
    """Xoá cache HF (không đụng token). Gọi sau mỗi phase dùng HF."""
    for root in HF_DIRS:
        for sub in subdirs:
            shutil.rmtree(root / sub, ignore_errors=True)
    free_memory()


def atomic_pickle_dump(obj, path):
    """Ghi file tạm rồi rename để tránh pickle hỏng khi kernel bị ngắt."""
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with open(tmp, 'wb') as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def atomic_json_dump(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with open(tmp, 'w') as f:
        json.dump(obj, f)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def load_pickle(path):
    with open(path, 'rb') as f:
        return pickle.load(f)


def is_oom(e):
    return 'out of memory' in str(e).lower()


print('RUN:', RUN_NAME, '| Z:', EMB_MODEL, '| prior:', REL_PRIOR, '| stop:', USE_STOP)
print('DEVICE:', DEVICE, '| GPUs:', torch.cuda.device_count(), '| HF_HOME:', HF_HOME)
report('khởi động')

# Phase 1 – Convert HF → shard nhỏ

In [ ]:
_SUFFIX_RE = re.compile(r'\s*\(([^()]*)\)\s*$')


def parse_name_cleaned(s):
    return _SUFFIX_RE.sub('', s).strip()


def convert_one(raw, keep_names=False):
    """keep_names=False cho train/val (không cần tên entity khi train → nhẹ hơn nhiều).
    keep_names=True cho test (cần tên để xuất triple đọc được)."""
    qid = str(raw['id'])
    question = raw['question']

    entities, relations = [], []
    ent2idx, rel2idx = {}, {}
    hs, rs, ts = [], [], []

    # Gold triples: normalize relation name giống graph
    gold_set = {(g[0], parse_name_cleaned(g[1]), g[2]) for g in raw['answer_subgraph']}
    labels = []

    for h, r, t in raw['graph']:
        h_idx = ent2idx.get(h)
        if h_idx is None:
            h_idx = len(entities)
            ent2idx[h] = h_idx
            entities.append(h)

        t_idx = ent2idx.get(t)
        if t_idx is None:
            t_idx = len(entities)
            ent2idx[t] = t_idx
            entities.append(t)

        r_name = parse_name_cleaned(r)
        r_idx = rel2idx.get(r_name)
        if r_idx is None:
            r_idx = len(relations)
            rel2idx[r_name] = r_idx
            relations.append(r_name)

        hs.append(h_idx)
        rs.append(r_idx)
        ts.append(t_idx)
        labels.append(float((h, r_name, t) in gold_set))

    q_ent = sorted({ent2idx[s] for s in raw['seed_entities']})   # KeyError nếu seed không có trong graph
    a_ent = [ent2idx[raw['answer_node']]]                        # KeyError nếu answer không có trong graph
    n_ent = len(entities)

    rec = {
        'id': qid,
        'question': question,
        'h': np.asarray(hs, dtype=np.int32),
        'r': np.asarray(rs, dtype=np.int32),
        't': np.asarray(ts, dtype=np.int32),
        'q_ent': q_ent,
        'a_ent': a_ent,
        'n_text': n_ent,
        'n_ent': n_ent,
        'relation_list': relations,
        'label': np.asarray(labels, dtype=np.float32),
    }
    if keep_names:
        rec['entity_names'] = entities
    return rec

In [ ]:
def stable_is_val(qid, val_ratio):
    """Chia train/val ổn định theo ID, không phụ thuộc thứ tự chạy."""
    if val_ratio <= 0:
        return False
    digest = hashlib.blake2b(str(qid).encode('utf-8'), digest_size=8).digest()
    return int.from_bytes(digest, 'big') / (2 ** 64) < val_ratio


# ---- manifest: nguồn sự thật duy nhất về các shard (dùng cho resume, embedding và train) ----
def _manifest_path(split):
    return OUT_DIR / f'manifest_{split}.json'


def load_manifest(split):
    p = _manifest_path(split)
    return json.load(open(p)) if p.exists() else None


def split_done(split):
    m = load_manifest(split)
    return bool(m and m.get('done'))


def get_shards(manifest_split, kind):
    """Danh sách shard (dict: file, kind, n_rec, n_rel). Với split 'train': kind ∈ {'train','val'}."""
    m = load_manifest(manifest_split)
    if m is None:
        raise FileNotFoundError(f'Chưa có manifest_{manifest_split}.json – chạy Phase 1 trước.')
    return [s for s in m['shards'] if s['kind'] == kind]


def convert_split_streaming(raw_ds, split_name, val_ratio=0.05, block_size=BLOCK_SIZE, keep_names=None):
    """Convert theo block `block_size` câu thô. Mỗi block ghi shard + cập nhật manifest
    → kernel chết giữa chừng thì chạy lại sẽ tiếp tục từ block đang dở.
    RAM tối đa ≈ 1 block."""
    keep_names = (split_name != 'train') if keep_names is None else keep_names
    mp = _manifest_path(split_name)
    st = load_manifest(split_name) or {'next_raw': 0, 'block_id': 0, 'shards': [],
                                       'converted': 0, 'skipped': {}}
    if st.get('done'):
        print(f'[{split_name}] đã convert xong trước đó.')
        return st

    def skip(reason):
        st['skipped'][reason] = st['skipped'].get(reason, 0) + 1

    n_total = len(raw_ds)
    pbar = tqdm(total=n_total, initial=st['next_raw'], desc=f'convert {split_name}')

    for start in range(st['next_raw'], n_total, block_size):
        end = min(start + block_size, n_total)
        buffers = {'train': [], 'val': [], 'test': []}

        for raw in raw_ds.select(range(start, end)):
            try:
                rec = convert_one(raw, keep_names=keep_names)
            except KeyError:
                skip('seed/answer không có trong graph')
                continue
            except Exception:
                skip('lỗi khác khi convert')
                continue

            if len(rec['h']) == 0:
                skip('graph rỗng'); continue
            if not rec['q_ent']:
                skip('không có seed'); continue
            if rec['label'].sum() == 0:
                skip('không có gold triple trong graph'); continue

            if split_name == 'train':
                kind = 'val' if stable_is_val(rec['id'], val_ratio) else 'train'
            else:
                kind = split_name
            buffers[kind].append(rec)
            st['converted'] += 1

        for kind, recs in buffers.items():
            if not recs:
                continue
            fname = f'{kind}_{st["block_id"]:05d}.pkl'
            atomic_pickle_dump(recs, OUT_DIR / fname)
            st['shards'].append({'file': fname, 'kind': kind, 'n_rec': len(recs),
                                 'n_rel': int(sum(len(r['relation_list']) for r in recs))})
        st['next_raw'] = end
        st['block_id'] += 1
        atomic_json_dump(st, mp)          # checkpoint sau MỖI block
        pbar.update(end - start)
        del buffers

    pbar.close()
    st['done'] = True
    atomic_json_dump(st, mp)
    print(f'[{split_name}] converted={st["converted"]:,} | skipped={st["skipped"]} | '
          f'shards={len(st["shards"])}')
    return st

## Reuse the processed data of v1 (optional)

Links the shards, manifests and GTE embeddings of `REUSE_DIR` (read-only Kaggle input) into `OUT_DIR` with symlinks, so nothing is copied and Phases 1 and 2 are skipped. The embedding folders are recreated in `OUT_DIR` with links to each file, because Phase 2b writes `prior.npy` next to them.
The LoRA weights are merged into BGE-M3 directly (no `peft`).

In [ ]:
def link_processed(src_dir, dst_dir=OUT_DIR):
    """symlink manifests, shards and lemb_<split>/* of a v1 output into dst_dir (as lemb_<Z_ENCODER>_<split>)"""
    src_dir, dst_dir = Path(src_dir), Path(dst_dir)
    n = 0
    for f in src_dir.iterdir():
        if f.is_file() and (f.suffix == '.pkl' or f.name.startswith('manifest_')):
            target = dst_dir / f.name
            if not target.exists():
                target.symlink_to(f); n += 1
    for split in ('train', 'val', 'test'):
        src_lemb = src_dir / f'lemb_{split}'
        if not (src_lemb / 'index.pkl').exists():
            print(f'  {src_lemb} thiếu index.pkl → Phase 2 sẽ build lại {split}'); continue
        dst_lemb = dst_dir / f'lemb_{Z_ENCODER}_{split}'
        dst_lemb.mkdir(parents=True, exist_ok=True)
        for f in src_lemb.iterdir():
            target = dst_lemb / f.name
            if not target.exists():
                target.symlink_to(f); n += 1
    print(f'linked {n} files from {src_dir}')


if REUSE_DIR and Path(REUSE_DIR).exists():
    if Z_ENCODER != 'gte':
        print('REUSE_DIR has GTE embeddings: only the shards are reused (Z_ENCODER != gte)')
    link_processed(REUSE_DIR) if Z_ENCODER == 'gte' else None
    if Z_ENCODER != 'gte':
        for f in Path(REUSE_DIR).iterdir():
            if f.is_file() and (f.suffix == '.pkl' or f.name.startswith('manifest_')) and not (OUT_DIR / f.name).exists():
                (OUT_DIR / f.name).symlink_to(f)
    print('manifest train done:', split_done('train'), '| test done:', split_done('test'))
elif REUSE_DIR:
    print(f'REUSE_DIR {REUSE_DIR} not found: building from HF')


In [ ]:
report('trước Phase 1')

if split_done('train') and split_done('test'):
    print('Phase 1 đã hoàn tất từ lần chạy trước → bỏ qua (không tải lại dataset).')
else:
    from datasets import load_dataset

    def load_gtsqa_hf(split='train'):
        ds = load_dataset('Graphcore/GTSQA', 'gtsqa-with-graphs', split=split, trust_remote_code=True)
        print(f'HF GTSQA {split}: {len(ds)} câu')
        return ds

    todo = [s for s in ('train', 'test') if not split_done(s)]
    raws = {s: load_gtsqa_hf(s) for s in todo}

    # Arrow đã sinh xong → file tải về (downloads/hub) không còn cần nữa: xoá sớm để nhả đĩa
    clear_hf_cache(subdirs=('datasets/downloads', 'hub'))
    report('đã xoá file tải về')

    for s in todo:
        convert_split_streaming(raws[s], s, val_ratio=0.05, block_size=BLOCK_SIZE)
        raws[s] = None
        free_memory()
        report(f'xong convert {s}')

    del raws
    free_memory()

# ---------- DỌN DẸP PHASE 1 ----------
clear_hf_cache()
report('SAU Phase 1 (đã dọn cache HF)')

# Phase 2 – Embedding (chỉ relation + question)

In [ ]:
STOPWORDS = {
    'what', 'which', 'who', 'whom', 'where', 'when', 'why', 'how',
    'a', 'an', 'the', 'and', 'or', 'but', 'if', 'because', 'as', 'until', 'while',
    'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into',
    'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from',
    'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further',
    'then', 'once', 'here', 'there', 'all', 'any', 'both', 'each', 'few', 'more',
    'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same',
    'so', 'than', 'too', 'very', 's', 't', 'can', 'will', 'just', 'don',
    'should', 'now', 'is', 'was', 'were', 'are', 'has', 'have', 'had', 'do', 'does', 'did'
}

SYNONYM_GROUPS = [
    {'speak', 'spoken', 'speech', 'language', 'languages'},
    {'born', 'birth', 'originated', 'origin'},
    {'died', 'death', 'deceased', 'burial', 'grave'},
    {'written', 'write', 'author', 'book', 'writer', 'subjects'},
    {'acted', 'actor', 'actress', 'starring', 'play', 'plays', 'cast', 'role', 'roles'},
    {'produced', 'producer'},
    {'directed', 'director'},
    {'created', 'creator', 'creative'},
    {'founded', 'founder', 'founding'},
    {'located', 'location', 'place', 'containedby', 'contains', 'adjoins', 'adjoin'},
    {'member', 'membership', 'participant', 'participated'},
    {'owned', 'owner', 'proprietor'},
    {'married', 'marriage', 'spouse', 'wife', 'husband'},
    {'parent', 'parents', 'children', 'child', 'son', 'daughter', 'father', 'mother'},
    {'nationality', 'citizen', 'country', 'nation'}
]


def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return [word for word in text.split() if word not in STOPWORDS]


# lru_cache: cặp (token câu hỏi, token relation) lặp lại rất nhiều lần -> nhanh hơn hàng chục lần, kết quả y hệt
@lru_cache(maxsize=500_000)
def get_token_similarity(q_token, r_token):
    if q_token == r_token:
        return 1.0
    for group in SYNONYM_GROUPS:
        if q_token in group and r_token in group:
            return 0.9
    min_len = min(len(q_token), len(r_token))
    if min_len >= 4:
        if q_token[:min_len] == r_token[:min_len]:
            return 0.9
        if q_token[:4] == r_token[:4]:
            return 0.8
    return 0.0


@lru_cache(maxsize=None)
def _rel_tokens(relation_name):
    """[(token_lower, weight)] của một relation (đã lọc stopword / token quá ngắn)."""
    parts = relation_name.split('.')
    n = len(parts)
    out = []
    for idx, part in enumerate(parts):
        weight = 1.0 if idx == n - 1 else (0.5 if idx == n - 2 else 0.2)
        for r_tok in part.split('_'):
            if len(r_tok) < 2:
                continue
            r_low = r_tok.lower()
            if r_low in STOPWORDS:
                continue
            out.append((r_low, weight))
    return tuple(out)


def lexical_score(question_tokens, relation_name):
    total_score, total_weight = 0.0, 0.0
    for r_tok, weight in _rel_tokens(relation_name):
        best = 0.0
        for q_tok in question_tokens:
            s = get_token_similarity(q_tok, r_tok)
            if s > best:
                best = s
        total_score += best * weight
        total_weight += weight
    return 0.0 if total_weight == 0.0 else total_score / total_weight

In [ ]:
class Embedder:
    """EMB_MODEL (gte-large-en-v1.5 or bge-m3) ở fp16, max_seq_length ngắn (câu hỏi / tên relation đều ngắn) → ít VRAM, nhanh.
    Gọi .close() để gỡ hẳn khỏi GPU trước khi train."""

    def __init__(self, model_name=EMB_MODEL, device=DEVICE, max_seq_length=128, fp16=True):
        self.device, self.st, self.model, self.tok, self.max_len = device, None, None, None, max_seq_length
        use_half = fp16 and str(device).startswith('cuda')
        try:
            from sentence_transformers import SentenceTransformer
            # gte-large-en-v1.5 BẮT BUỘC trust_remote_code=True
            self.st = SentenceTransformer(model_name, device=device, trust_remote_code='gte' in model_name)
            self.st.max_seq_length = max_seq_length
            if use_half:
                self.st.half()
        except Exception as e:
            print('SentenceTransformer lỗi → dùng transformers thuần:', repr(e)[:300])
            self.st = None
            from transformers import AutoTokenizer, AutoModel
            self.tok = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModel.from_pretrained(model_name, trust_remote_code='gte' in model_name)
            if use_half:
                self.model = self.model.half()
            self.model = self.model.to(device).eval()

    @torch.no_grad()
    def _encode(self, texts, bs):
        if self.st is not None:
            return self.st.encode(texts, batch_size=bs, show_progress_bar=False,
                                  convert_to_numpy=True, normalize_embeddings=True)
        outs = []
        for i in range(0, len(texts), bs):
            inputs = self.tok(texts[i:i + bs], padding=True, truncation=True,
                              max_length=self.max_len, return_tensors='pt').to(self.device)
            cls = self.model(**inputs).last_hidden_state[:, 0]          # gte and bge-m3 dense: CLS pooling
            outs.append(F.normalize(cls.float(), dim=-1).cpu().numpy())
        return np.concatenate(outs, 0)

    def __call__(self, texts, batch_size=BATCH_SIZE_EMB):
        texts = list(texts)
        bs = batch_size
        while True:
            try:
                return np.asarray(self._encode(texts, bs), dtype=np.float32)
            except RuntimeError as e:
                if not is_oom(e) or bs <= 4:
                    raise
                bs //= 2
                free_memory()
                print(f'OOM khi embed → giảm batch xuống {bs}')

    def close(self):
        self.st = self.model = self.tok = None
        free_memory()

In [ ]:
def encode_to_npy(texts, n, path, encode_fn, batch_size=256, dtype=np.float16, dim=EMB_DIM):
    """Encode theo batch và ghi thẳng vào .npy (memmap). Không giữ toàn bộ embedding trong RAM.
    Nếu file đã có đúng shape thì bỏ qua (resume). Ghi vào .part rồi rename."""
    path = Path(path)
    if path.exists():
        try:
            arr = np.load(path, mmap_mode='r')
            ok = tuple(arr.shape) == (n, dim)
            del arr
        except Exception:
            ok = False
        if ok:
            print(f'  {path.name}: đã có → bỏ qua')
            return
    tmp = Path(str(path) + '.part')

    if n == 0:
        with open(tmp, 'wb') as f:
            np.save(f, np.empty((0, dim), dtype=dtype))
        os.replace(tmp, path)
        return

    texts = iter(texts)
    output = np.lib.format.open_memmap(tmp, mode='w+', dtype=dtype, shape=(n, dim))
    offset = 0
    pbar = tqdm(total=n, desc=f'Embedding {path.name}')
    while offset < n:
        batch = []
        for _ in range(min(batch_size, n - offset)):
            try:
                batch.append(next(texts))
            except StopIteration:
                break
        if not batch:
            break
        emb = encode_fn(batch)
        if emb.ndim != 2 or emb.shape != (len(batch), dim):
            raise ValueError(f'Embedding shape lạ: {emb.shape}')
        output[offset:offset + len(batch)] = emb
        offset += len(batch)
        pbar.update(len(batch))
        del emb, batch
    pbar.close()
    if offset != n:
        raise RuntimeError(f'Encoded {offset}/{n} texts')
    output.flush()
    del output
    os.replace(tmp, path)
    gc.collect()


def build_lemb_compact(shards, split, encode_fn, batch_size=256, force=False):
    """Compact RelationOnlyEmb (đọc lần lượt từng shard, mỗi lúc chỉ 1 shard trong RAM):
      q.npy, rel_unique.npy, rel_ids.npy, lex.npy, index.pkl (ghi cuối cùng = dấu hiệu hoàn tất)"""
    out = OUT_DIR / f'lemb_{Z_ENCODER}_{split}'
    out.mkdir(parents=True, exist_ok=True)
    if (out / 'index.pkl').exists() and not force:
        print(f'[{split}] đã có index.pkl → bỏ qua (force=True để build lại)')
        return out

    n_questions = sum(s['n_rec'] for s in shards)
    n_mentions = sum(s['n_rel'] for s in shards)
    print(f'[{split}] questions: {n_questions:,} | relation mentions: {n_mentions:,}')

    rel_ids = np.lib.format.open_memmap(out / 'rel_ids.npy', mode='w+', dtype=np.int32, shape=(n_mentions,))
    lex = np.lib.format.open_memmap(out / 'lex.npy', mode='w+', dtype=np.float32, shape=(n_mentions,))

    relation_to_id, unique_relations, questions, index = {}, [], [], {}
    offset, row = 0, 0

    # ---- 1. Quét shard: vocab relation, rel_ids, lex, index ----
    for s in tqdm(shards, desc=f'index/lex {split}'):
        recs = load_pickle(OUT_DIR / s['file'])
        for rec in recs:
            q_words = list(dict.fromkeys(clean_text(rec['question'])))
            relations = rec['relation_list']
            start = offset
            ids = np.empty(len(relations), dtype=np.int32)
            lx = np.empty(len(relations), dtype=np.float32)
            for j, relation in enumerate(relations):
                rid = relation_to_id.get(relation)
                if rid is None:
                    rid = len(unique_relations)
                    relation_to_id[relation] = rid
                    unique_relations.append(relation)
                ids[j] = rid
                lx[j] = lexical_score(q_words, relation)
            offset += len(relations)
            rel_ids[start:offset] = ids
            lex[start:offset] = lx
            index[rec['id']] = (row, start, offset, rec['n_text'])
            questions.append(rec['question'])
            row += 1
        del recs

    assert offset == n_mentions and row == n_questions, (offset, n_mentions, row, n_questions)
    rel_ids.flush(); lex.flush()
    del rel_ids, lex
    gc.collect()

    n_unique = len(unique_relations)
    print(f'  Unique relations: {n_unique:,} (dedup {n_mentions / max(n_unique, 1):.1f}x)')

    # ---- 2. Encode câu hỏi, 3. encode relation unique ----
    encode_to_npy(iter(questions), n_questions, out / 'q.npy', encode_fn, batch_size)
    encode_to_npy(iter(unique_relations), n_unique, out / 'rel_unique.npy', encode_fn, batch_size)

    # ---- 4. index.pkl cuối cùng ----
    atomic_pickle_dump(index, out / 'index.pkl')
    print(f'  Saved: {out}')

    del unique_relations, relation_to_id, index, questions
    free_memory()
    return out

In [ ]:
report('trước Phase 2')
_need = [s for s in ('train', 'val', 'test') if not (OUT_DIR / f'lemb_{Z_ENCODER}_{s}' / 'index.pkl').exists()]
if not _need:
    print('Phase 2: embeddings đã có cho cả 3 split → bỏ qua (không nạp encoder)')
else:

    embedder = Embedder()
    report(f'đã nạp {EMB_MODEL}')
    try:
        build_lemb_compact(get_shards('train', 'train'), 'train', embedder, batch_size=256)
        free_memory()
        build_lemb_compact(get_shards('train', 'val'), 'val', embedder, batch_size=256)
        free_memory()
        build_lemb_compact(get_shards('test', 'test'), 'test', embedder, batch_size=256)
    finally:
        # ---------- DỌN DẸP PHASE 2: gỡ model embedding khỏi GPU + xoá cache model ----------
        embedder.close()
        del embedder
        clear_hf_cache()
        report('SAU Phase 2 (đã gỡ encoder, xoá cache)')


# Phase 2b – Fine-tuned BGE-M3 similarity (REL_PRIOR='bge_ft')

Cosine of the question and each relation under BGE-M3 + LoRA (fine-tuned on WebQSP, as for WebQSP / CWQ): CLS pooling, L2 norm, questions ≤ 96 tokens, relation names in the *split* format (`.` and `_` → spaces) ≤ 48 tokens. Stored per relation mention, aligned with `rel_ids.npy`.

In [ ]:
def split_relation(r):
    return r.replace('.', ' ').replace('_', ' ')


def load_bge_ft(base=BGE_BASE, adapter=BGE_ADAPTER, device=DEVICE):
    """BGE-M3 with the LoRA adapter merged into its weights, without peft (recent peft versions fail on Kaggle's
    torchao). For every adapted Linear: W <- W + (lora_alpha / r) * B @ A, which is what peft computes at inference."""
    from transformers import AutoModel, AutoTokenizer
    from safetensors.torch import load_file
    tok = AutoTokenizer.from_pretrained(base)
    model = AutoModel.from_pretrained(base, torch_dtype=torch.float32)
    cfg = json.load(open(f'{adapter}/adapter_config.json'))
    scale = cfg['lora_alpha'] / cfg['r']
    lora = load_file(f'{adapter}/adapter_model.safetensors')
    modules = dict(model.named_modules())
    n = 0
    for key in lora:
        if not key.endswith('lora_A.weight'):
            continue
        name = key[len('base_model.model.'):-len('.lora_A.weight')]
        A, B = lora[key].float(), lora[key.replace('lora_A', 'lora_B')].float()
        with torch.no_grad():
            modules[name].weight += scale * (B @ A)
        n += 1
    assert n == len(lora) // 2 and n > 0, f'merged {n} of {len(lora) // 2} LoRA modules'
    print(f'BGE-M3: merged {n} LoRA modules (scale {scale})')
    if str(device).startswith('cuda'):
        model = model.half()
    return tok, model.to(device).eval()


@torch.no_grad()
def bge_encode(tok, model, texts, max_len, bs=128, device=DEVICE):
    """[n, 1024] float32, CLS pooling + L2 norm; length-sorted batches"""
    order = np.argsort([len(t) for t in texts])
    out = np.empty((len(texts), 1024), dtype=np.float32)
    for i in tqdm(range(0, len(texts), bs), desc=f'BGE ({len(texts):,})', leave=False):
        idx = order[i:i + bs]
        b = tok([texts[j] for j in idx], max_length=max_len, truncation=True, padding=True, return_tensors='pt').to(device)
        with torch.autocast('cuda', dtype=torch.float16, enabled=str(device).startswith('cuda')):
            h = model(**b).last_hidden_state[:, 0]
        out[idx] = F.normalize(h.float(), dim=-1).cpu().numpy()
    return out


def build_bge_prior(shards, split, tok, model, force=False):
    """lemb_<Z>_<split>/prior.npy [n_mentions] float16 = cos(question, relation) under the fine-tuned BGE-M3.
    Re-scans the shards in the order of build_lemb_compact, so questions / relations line up with index.pkl / rel_ids."""
    d = OUT_DIR / f'lemb_{Z_ENCODER}_{split}'
    if (d / 'prior.npy').exists() and not force:
        print(f'[{split}] prior.npy đã có → bỏ qua'); return
    index = load_pickle(d / 'index.pkl')
    rel_ids = np.load(d / 'rel_ids.npy', mmap_mode='r')
    questions, rows, relation_to_id, unique = [], [], {}, []
    for s in tqdm(shards, desc=f'scan {split}'):
        for rec in load_pickle(OUT_DIR / s['file']):
            questions.append(rec['question']); rows.append(index[rec['id']])
            for r in rec['relation_list']:
                if r not in relation_to_id:
                    relation_to_id[r] = len(unique); unique.append(r)
    Q = bge_encode(tok, model, questions, 96)
    R = bge_encode(tok, model, [split_relation(r) for r in unique], 48)
    prior = np.lib.format.open_memmap(d / 'prior.npy.part', mode='w+', dtype=np.float16, shape=(len(rel_ids),))
    for k, (q_row, start, end, _) in enumerate(rows):
        prior[start:end] = (R[np.asarray(rel_ids[start:end])] @ Q[k]).astype(np.float16)
    prior.flush(); del prior
    os.replace(d / 'prior.npy.part', d / 'prior.npy')
    print(f'[{split}] prior.npy: {len(rel_ids):,} mentions, {len(unique):,} relations')


if REL_PRIOR == 'bge_ft':
    report('trước Phase 2b')
    tok_bge, model_bge = load_bge_ft()
    try:
        build_bge_prior(get_shards('train', 'train'), 'train', tok_bge, model_bge); free_memory()
        build_bge_prior(get_shards('train', 'val'), 'val', tok_bge, model_bge); free_memory()
        build_bge_prior(get_shards('test', 'test'), 'test', tok_bge, model_bge)
    finally:
        del tok_bge, model_bge
        clear_hf_cache()
        report('SAU Phase 2b')


# Phase 3 – Train RBF

In [ ]:
import torch.nn.functional as F
try:
    from torch_scatter import scatter_max
except Exception:
    # Không cần cài torch_scatter: bản thay thế thuần PyTorch (src 2D [A, C], dim=1, index 1D [C]).
    # Trả về (max, argmax) giống torch_scatter; ô rỗng -> (0, C). Gradient đi qua phần tử được chọn.
    def scatter_max(src, index, dim=-1, dim_size=None):
        assert src.dim() == 2 and index.dim() == 1 and dim in (1, -1)
        A, C = src.shape
        if C == 0:
            return (src.new_zeros(A, dim_size),
                    torch.zeros(A, dim_size, dtype=torch.long, device=src.device))
        with torch.no_grad():
            idx = index[None, :].expand(A, C)
            mx = torch.full((A, dim_size), torch.finfo(src.dtype).min, dtype=src.dtype, device=src.device)
            mx.scatter_reduce_(1, idx, src, reduce='amax', include_self=True)
            hit = src == mx.gather(1, idx)
            pos = torch.arange(C, device=src.device)[None, :].expand(A, C)
            pos = torch.where(hit, pos, torch.full_like(pos, C))
            arg = torch.full((A, dim_size), C, dtype=torch.long, device=src.device)
            arg.scatter_reduce_(1, idx, pos, reduce='amin', include_self=True)
        has = arg < C
        out = src.gather(1, arg.clamp(max=C - 1)).masked_fill(~has, 0.0)
        return out, arg

NEG = -1e4                       # log of "unreachable"
COMPLETION_STEPS = 2             # mm: longest continuation after a triple


class RBF(nn.Module):
    def __init__(self, emb=1024, hidden=256, T=3, mm=False, extra_prior=0, stop=False):
        """emb: text-embedding size; T: propagation steps; mm: complete-path scoring (set True at test time);
        extra_prior: number of extra per-relation prior columns read from x['rel_prior'] (BGE: 2);
        stop: add the stop factor phi (see below; the best configuration trains it jointly with pi, --stop)"""
        super().__init__()
        self.T, self.mm, self.extra_prior = T, mm, extra_prior
        n_in = 3 * emb + 2 + extra_prior
        self.pi = nn.Sequential(nn.Linear(n_in, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 2))
        # stop factor phi(stop | q, r): probability that a walk whose LAST step is relation r (forward / reverse) ends
        # there, i.e. that r is the question's answer relation. Same inputs as pi, so it still depends on (q, r) only.
        # Used by mm: a complete walk pays log phi of its last step instead of stopping for free.
        self.stop = nn.Sequential(nn.Linear(n_in, hidden), nn.ReLU(), nn.Linear(hidden, 2)) if stop else None

    def relation_inputs(self, x):
        q = x['q'].expand(len(x['rel']), -1)
        parts = [q, x['rel'], q * x['rel'], x['lex'][:, None], F.cosine_similarity(x['rel'], q, dim=-1)[:, None]]
        if self.extra_prior:
            parts.append(x['rel_prior'])
        return torch.cat(parts, 1)

    def relation_logits(self, x):
        """[n_relations, 2] logits of pi(r | q): column 0 = forward step, column 1 = reverse step"""
        return self.pi(self.relation_inputs(x))

    def stop_logits(self, x):
        """[n_relations, 2] logits of phi(stop | q, r) for a last step forward / reverse"""
        return self.stop(self.relation_inputs(x))

    def forward(self, x):
        """-> final log-score of every triple [n_triples]"""
        n, h, t, r = x['n_ent'], x['h'], x['t'], x['r']
        logit = self.relation_logits(x)
        log_pi_fwd, log_pi_rev = F.logsigmoid(logit[:, 0])[r], F.logsigmoid(logit[:, 1])[r]     # per triple
        anchors = x['topic'][:, 1].nonzero().reshape(-1)
        log_p = torch.full((max(len(anchors), 1), n), NEG, device=h.device)                    # [anchors, nodes]
        if len(anchors):
            log_p[torch.arange(len(anchors), device=h.device), anchors] = 0.0
        path_fwd, path_rev, log_p = propagate_non_backtracking(log_p, h, t, r, log_pi_fwd, log_pi_rev, n, self.T)
        if self.mm:
            log_stop = F.logsigmoid(self.stop_logits(x)) if self.stop is not None else None
            after_fwd, after_rev = best_completion(h, t, r, log_pi_fwd, log_pi_rev, n, n_text=len(x['ent']),
                                                   log_stop_fwd=None if log_stop is None else log_stop[:, 0][r],
                                                   log_stop_rev=None if log_stop is None else log_stop[:, 1][r])
            path_fwd, path_rev = (path_fwd + after_fwd).clamp(min=NEG), (path_rev + after_rev).clamp(min=NEG)
        path = torch.maximum(path_fwd, path_rev)                                                # [anchors, triples]
        union = path.max(0).values
        reached_by_all = log_p.sum(0)                                                           # [nodes]
        inter = torch.maximum(path_fwd + reached_by_all[t] - log_p[:, t],
                              path_rev + reached_by_all[h] - log_p[:, h]).max(0).values
        return torch.logaddexp(union, inter)


# ------------------------------------------------------------------ non-backtracking max-product (Viterbi top-2)
def _top2_merge(best, best_tag, second, second_tag, candidates, candidate_tags, destinations, n):
    """Update each node's (best score, its last-step tag, best score with another tag, its tag) with candidate steps
    `candidates` [A, C] arriving at `destinations` [C] with tags `candidate_tags` [C]."""
    m1, arg = scatter_max(candidates, destinations, dim=1, dim_size=n)
    has = arg < candidates.shape[1]
    m1 = m1.masked_fill(~has, NEG)
    t1 = torch.where(has, candidate_tags[arg.clamp(max=candidates.shape[1] - 1)], torch.full_like(arg, -2))
    take = m1 > best
    new_best, new_tag = torch.where(take, m1, best), torch.where(take, t1, best_tag)
    others = candidates.masked_fill(candidate_tags[None, :] == new_tag[:, destinations], NEG)
    m2, arg2 = scatter_max(others, destinations, dim=1, dim_size=n)
    has2 = arg2 < candidates.shape[1]
    m2 = m2.masked_fill(~has2, NEG)
    t2 = torch.where(has2, candidate_tags[arg2.clamp(max=candidates.shape[1] - 1)], torch.full_like(arg2, -2))
    old_best = torch.where(best_tag != new_tag, best, torch.full_like(best, NEG))   # old best survives as 2nd
    old_second = torch.where(second_tag != new_tag, second, torch.full_like(second, NEG))
    new_second, new_second_tag = m2, t2
    for value, tag in ((old_best, best_tag), (old_second, second_tag)):
        better = value > new_second
        new_second, new_second_tag = torch.where(better, value, new_second), torch.where(better, tag, new_second_tag)
    return new_best.clamp(min=NEG), new_tag, new_second.clamp(min=NEG), new_second_tag


def _best_without_tag(best, best_tag, second, nodes, tag):
    """best score at `nodes` [E] over walks whose last step is not `tag` [E]"""
    return torch.where(best_tag[:, nodes] == tag[None, :], second[:, nodes], best[:, nodes])


def propagate_non_backtracking(log_p, h, t, r, log_pi_fwd, log_pi_rev, n, T):
    """T steps of non-backtracking max-product from the anchor rows of log_p [A, n].
    -> (score of crossing each triple forwards [A, E], backwards [A, E], node scores [A, n])"""
    best, second = log_p, torch.full_like(log_p, NEG)
    best_tag = torch.full(log_p.shape, -1, dtype=torch.long, device=log_p.device)
    second_tag = torch.full_like(best_tag, -2)
    tag_fwd, tag_rev = 2 * r, 2 * r + 1                       # tag of a forward / reverse step over relation r
    destinations, tags = torch.cat([t, h]), torch.cat([tag_fwd, tag_rev])
    for _ in range(T):
        step_fwd = _best_without_tag(best, best_tag, second, h, tag_rev) + log_pi_fwd   # h -> t, not right after t -> h on r
        step_rev = _best_without_tag(best, best_tag, second, t, tag_fwd) + log_pi_rev   # t -> h, not right after h -> t on r
        best, best_tag, second, second_tag = _top2_merge(best, best_tag, second, second_tag,
                                                         torch.cat([step_fwd, step_rev], 1), tags, destinations, n)
    return (_best_without_tag(best, best_tag, second, h, tag_rev) + log_pi_fwd,
            _best_without_tag(best, best_tag, second, t, tag_fwd) + log_pi_rev, best)


def best_completion(h, t, r, log_pi_fwd, log_pi_rev, n, n_text, log_stop_fwd=None, log_stop_rev=None):
    """log score of the best non-backtracking continuation (<= COMPLETION_STEPS steps) after a forward step into t /
    a reverse step into h that stops at a text entity (nodes 0..n_text-1); anchor-independent -> [1, E] each.
    Without a stop factor, stopping at a text entity is free. With one (log_stop_* per triple = log phi of the step
    over that triple), a walk may stop right after a step into a text entity and pays log phi of that step."""
    best = torch.full((1, n), NEG, device=h.device)
    if log_stop_fwd is None:
        best[:, :n_text] = 0.0                                # stopping at a text entity is free
        stop_fwd = stop_rev = None
    else:                                                     # stopping right after the step over the triple
        stop_fwd = torch.where(t < n_text, log_stop_fwd, torch.full_like(log_stop_fwd, NEG))[None, :]
        stop_rev = torch.where(h < n_text, log_stop_rev, torch.full_like(log_stop_rev, NEG))[None, :]
    best_tag = torch.full(best.shape, -1, dtype=torch.long, device=h.device)
    second, second_tag = torch.full_like(best, NEG), torch.full_like(best_tag, -2)
    tag_fwd, tag_rev = 2 * r, 2 * r + 1                       # tag = the continuation's first step
    destinations, tags = torch.cat([h, t]), torch.cat([tag_fwd, tag_rev])
    after = lambda node, tag, stop: (_best_without_tag(best, best_tag, second, node, tag) if stop is None
                                     else torch.maximum(_best_without_tag(best, best_tag, second, node, tag), stop))
    for _ in range(COMPLETION_STEPS):
        step_fwd = after(t, tag_rev, stop_fwd) + log_pi_fwd   # u = h continues forward to t (and stops there or goes on)
        step_rev = after(h, tag_fwd, stop_rev) + log_pi_rev   # u = t continues back to h
        best, best_tag, second, second_tag = _top2_merge(best, best_tag, second, second_tag,
                                                         torch.cat([step_fwd, step_rev], 1), tags, destinations, n)
    return after(t, tag_rev, stop_fwd), after(h, tag_fwd, stop_rev)


def log_to_logit(log_m):
    """logit of a probability given as log m (m <= 1)"""
    log_m = log_m.clamp(max=-1e-6)
    return log_m - torch.log(-torch.expm1(log_m))


## Load data (stream theo shard) và train

In [ ]:
class RelationOnlyEmb:
    """Embedding dạng compact, đọc bằng memmap – không nạp vào RAM."""

    def __init__(self, d):
        with open(f'{d}/index.pkl', 'rb') as f:
            self.index = pickle.load(f)
        self.q = np.load(f'{d}/q.npy', mmap_mode='r')
        self.rel = np.load(f'{d}/rel_unique.npy', mmap_mode='r')
        self.rel_ids = np.load(f'{d}/rel_ids.npy', mmap_mode='r')
        self.lex = np.load(f'{d}/lex.npy', mmap_mode='r')
        p = Path(d) / 'prior.npy'
        self.prior = np.load(p, mmap_mode='r') if p.exists() else None

    def get(self, qid):
        q_row, start, end, n_text = self.index[qid]
        q = np.array(self.q[q_row:q_row + 1], dtype=np.float32, copy=True)
        ids = np.asarray(self.rel_ids[start:end])
        rel = np.asarray(self.rel[ids], dtype=np.float32)
        return torch.from_numpy(q), torch.empty((n_text, 0)), torch.from_numpy(rel)

    def get_lex(self, qid):
        _, start, end, _ = self.index[qid]
        return np.array(self.lex[start:end], dtype=np.float32, copy=True)

    def get_prior(self, qid):
        """[n_rel, 2]: (fine-tuned BGE cosine, cosine - the question's max cosine), as rel_prior 'bge' on WebQSP / CWQ"""
        _, start, end, _ = self.index[qid]
        s = np.array(self.prior[start:end], dtype=np.float32, copy=True)
        return np.stack([s, s - s.max()], 1) if len(s) else np.zeros((0, 2), np.float32)

    def close(self):
        self.q = self.rel = self.rel_ids = self.lex = self.prior = self.index = None


class ShardStream:
    """Duyệt record theo shard, mỗi lúc chỉ giữ `chunk_shards` shard trong RAM.
    Shuffle = xáo thứ tự shard + xáo record trong chunk (mỗi epoch seed khác nhau)."""

    # train/val không cần question/relation_list/entity_names nữa (embedding đã build) → bỏ cho nhẹ RAM
    SLIM_DROP = ('question', 'relation_list', 'entity_names')

    def __init__(self, shards, shuffle=True, chunk_shards=4, seed=0, slim=True):
        self.files = [OUT_DIR / s['file'] for s in shards]
        self.n = sum(s['n_rec'] for s in shards)
        self.shuffle, self.chunk, self.seed, self.slim = shuffle, max(1, chunk_shards), seed, slim

    def __len__(self):
        return self.n

    def epoch(self, epoch=0):
        rng = random.Random(self.seed * 100003 + epoch)
        files = list(self.files)
        if self.shuffle:
            rng.shuffle(files)
        for i in range(0, len(files), self.chunk):
            recs = []
            for f in files[i:i + self.chunk]:
                recs.extend(load_pickle(f))
            if self.shuffle:
                rng.shuffle(recs)
            for rec in recs:
                if self.slim:
                    for k in self.SLIM_DROP:
                        rec.pop(k, None)
                yield rec
            del recs


def stop_targets(rec, n_rel, device):
    """targets of the stop factor phi, [n_relations, 2] (forward / reverse last step): label 1 for the step of a
    labelled triple that enters an answer; mask 1 for every step that enters a named entity (all GTSQA entities)"""
    h, r, t, n_text = rec['h'], rec['r'], rec['t'], rec['n_text']
    answers = np.zeros(rec['n_ent'], dtype=bool); answers[list(rec['a_ent'])] = True
    pos = np.nonzero(rec['label'])[0]
    label, mask = np.zeros((n_rel, 2), np.float32), np.zeros((n_rel, 2), np.float32)
    mask[r[t < n_text], 0] = 1; mask[r[h < n_text], 1] = 1
    label[r[pos][answers[t[pos]]], 0] = 1; label[r[pos][answers[h[pos]]], 1] = 1
    return {'stop_label': torch.from_numpy(label).to(device), 'stop_mask': torch.from_numpy(mask).to(device)}


def to_device(rec, emb, device):
    q, ent, rel = emb.get(rec['id'])
    topic = torch.zeros(rec['n_ent'], dtype=torch.long)
    topic[rec['q_ent']] = 1
    as_long = lambda a: torch.from_numpy(a.astype(np.int64, copy=False)).to(device)
    lex = torch.from_numpy(emb.get_lex(rec['id']))
    prior = torch.from_numpy(emb.get_prior(rec['id'])).to(device) if REL_PRIOR != 'none' else None
    return {
        'h': as_long(rec['h']), 'r': as_long(rec['r']), 't': as_long(rec['t']),
        'n_ent': rec['n_ent'],
        'q': q.to(device), 'ent': ent.to(device), 'rel': rel.to(device),
        'topic': F.one_hot(topic, 2).float().to(device),
        'lex': lex.to(device),
        'label': torch.from_numpy(rec['label'].astype(np.float32, copy=False)).to(device),
        'rel_prior': prior,
        **stop_targets(rec, rel.shape[0], device),
    }


def stop_loss(model, x):
    """BCE of phi over the steps that can end a path; positives and negatives weighted equally (as on WebQSP / CWQ)"""
    mask, label = x['stop_mask'] > 0, x['stop_label']
    if not (label[mask] > 0).any():
        return 0.0
    bce = F.binary_cross_entropy_with_logits(model.stop_logits(x), label, reduction='none')
    pos, neg = mask & (label > 0), mask & (label == 0)
    return 0.5 * bce[pos].mean() + (0.5 * bce[neg].mean() if neg.any() else 0.0)


def effective_number_weight(labels):
    n_pos = int(labels.sum().item())
    n_neg = len(labels) - n_pos
    if n_pos == 0:
        return None
    effective = lambda n: (1.0 - 0.999 ** n) / (1.0 - 0.999)
    return torch.tensor(effective(n_neg) / (effective(n_pos) + 1e-8), device=labels.device)


class Scorer(nn.Module):
    def __init__(self, rbf):
        super().__init__()
        self.rbf = rbf

    def forward(self, x):
        return log_to_logit(self.rbf(x) - 0.6931471805599453)


@torch.no_grad()
def score_record(scorer, rec, emb, device=DEVICE):
    """Điểm của mọi triple (CPU tensor). Nếu GPU hết VRAM → tính lại trên CPU thay vì chết."""
    oom = False
    try:
        return scorer(to_device(rec, emb, device)).cpu()
    except RuntimeError as e:
        if not is_oom(e):
            raise
        oom = True
    free_memory()
    cpu_scorer = copy.deepcopy(scorer).cpu()
    return cpu_scorer(to_device(rec, emb, 'cpu'))


@torch.no_grad()
def validate(scorer, stream, emb, device=DEVICE, ks=(50, 100)):
    """answer recall and recall of the labelled triples at ks"""
    scorer.eval()
    metrics = defaultdict(list)
    for rec in tqdm(stream.epoch(0), total=len(stream), desc='validate', leave=False):
        scores = score_record(scorer, rec, emb, device)
        order = torch.argsort(scores, descending=True).numpy()
        a_ent = set(rec['a_ent'])
        positive = rec['label'] > 0
        for k in ks:
            top = order[:k]
            ents = set(rec['h'][top].tolist()) | set(rec['t'][top].tolist())
            metrics[f'ansR@{k}'].append(len(a_ent & ents) / max(len(a_ent), 1))
            if positive.any():
                metrics[f'triple_recall@{k}'].append(positive[top].sum() / positive.sum())
    scorer.train()
    return {k: float(np.mean(v)) for k, v in metrics.items()}


In [ ]:
def train_rbf(
    train_shards, val_shards,
    lemb_train=OUT_DIR / f'lemb_{Z_ENCODER}_train',
    lemb_val=OUT_DIR / f'lemb_{Z_ENCODER}_val',
    out_dir=OUT_DIR / 'runs' / RUN_NAME,
    epochs=1000,
    patience=10,
    lr=1e-3,
    seed=42,
    use_stop=USE_STOP,
    chunk_shards=4,        # số shard giữ trong RAM cùng lúc (giảm nếu thiếu RAM, tăng để shuffle tốt hơn)
    max_hours=None,        # dừng êm sau N giờ (Kaggle giới hạn session) – chạy lại sẽ resume
    resume=True,
):
    t_start = time.time()
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    train_stream = ShardStream(train_shards, shuffle=True, chunk_shards=chunk_shards, seed=seed)
    val_stream = ShardStream(val_shards, shuffle=False, chunk_shards=1)
    emb_train, emb_val = RelationOnlyEmb(lemb_train), RelationOnlyEmb(lemb_val)
    print(f'train={len(train_stream):,} | val={len(val_stream):,} câu')

    model = RBF(emb=EMB_DIM, hidden=256, T=3, mm=False, extra_prior=2 if REL_PRIOR != 'none' else 0, stop=use_stop).to(DEVICE)
    scorer = Scorer(model).to(DEVICE)
    opt = torch.optim.Adam(scorer.parameters(), lr=lr)

    best, no_gain, log, start_epoch = -1.0, 0, [], 0
    last_path, best_path = out_dir / 'last.pt', out_dir / 'best.pt'
    if resume and last_path.exists():
        ck = torch.load(last_path, map_location=DEVICE, weights_only=False)
        scorer.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
        best, no_gain, log, start_epoch = ck['best'], ck['no_gain'], ck['log'], ck['epoch'] + 1
        print(f'>> Resume từ epoch {start_epoch} (best {SELECT_METRIC}={best:.4f}, no_gain={no_gain})')
        del ck

    for epoch in range(start_epoch, epochs):
        t0 = time.time()
        scorer.train()
        total_loss, n_done, n_oom = 0.0, 0, 0

        for rec in tqdm(train_stream.epoch(epoch), total=len(train_stream), desc=f'epoch {epoch}'):
            oom = False
            x = loss = None
            try:
                x = to_device(rec, emb_train, DEVICE)
                weight = effective_number_weight(x['label'])
                loss = F.binary_cross_entropy_with_logits(scorer(x), x['label'], pos_weight=weight)
                if use_stop:
                    loss = loss + stop_loss(model, x)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
                total_loss += loss.item()
                n_done += 1
            except RuntimeError as e:
                if not is_oom(e):
                    raise
                oom = True
            if oom:                                   # graph quá lớn cho VRAM → bỏ qua mẫu này
                n_oom += 1
                x = loss = None
                opt.zero_grad(set_to_none=True)
                free_memory()
            del x, loss

        # bật mm khi validate (giống infer)
        model.mm = True
        metrics = validate(scorer, val_stream, emb_val, DEVICE)
        model.mm = False

        sel = metrics[SELECT_METRIC]
        avg_loss = total_loss / max(n_done, 1)
        print(f'epoch {epoch}: loss={avg_loss:.4f}  val {SELECT_METRIC}={sel:.4f}  ansR@50={metrics["ansR@50"]:.4f}  '
              f'(oom_skipped={n_oom}, {time.time() - t0:.0f}s)')
        log.append({'epoch': epoch, 'loss': avg_loss, 'oom_skipped': n_oom, **metrics})

        if sel > best:
            best, no_gain = sel, 0
            torch.save(scorer.state_dict(), best_path)
            print('  → saved best')
        else:
            no_gain += 1

        # checkpoint mỗi epoch (ghi tạm rồi rename)
        tmp = out_dir / 'last.pt.tmp'
        torch.save({'model': scorer.state_dict(), 'opt': opt.state_dict(), 'epoch': epoch,
                    'best': best, 'no_gain': no_gain, 'log': log}, tmp)
        os.replace(tmp, last_path)
        json.dump({'config': {'Z_ENCODER': Z_ENCODER, 'REL_PRIOR': REL_PRIOR, 'USE_STOP': use_stop, 'SELECT_METRIC': SELECT_METRIC,
                              'lr': lr, 'seed': seed, 'patience': patience},
                   'log': log, f'best_{SELECT_METRIC}': best}, open(out_dir / 'result.json', 'w'), indent=1)
        free_memory()

        if no_gain >= patience:
            print('early stop')
            break
        if max_hours and (time.time() - t_start) / 3600 > max_hours:
            print(f'Hết ngân sách {max_hours}h → dừng êm. Chạy lại cell này để resume.')
            break

    # ---------- DỌN DẸP PHASE 3 ----------
    emb_train.close(); emb_val.close()
    del opt, train_stream, val_stream
    if best_path.exists():                            # dùng trọng số TỐT NHẤT, không phải epoch cuối
        scorer.load_state_dict(torch.load(best_path, map_location=DEVICE))
        print('Đã nạp lại best.pt')
    free_memory()
    print(f'Done. best {SELECT_METRIC} =', best)
    return scorer

In [ ]:
report('trước Phase 3')
scorer = train_rbf(
    get_shards('train', 'train'),
    get_shards('train', 'val'),
    max_hours=10,          # để chừa thời gian cho test; chạy lại cell này để resume nếu session bị ngắt
)
report('SAU Phase 3')

# Phase 4 – Inference / Test

In [ ]:
@torch.no_grad()
def retrieve_topk(scorer, stream, emb, device=DEVICE, k=TEST_TOPK):
    """Stream từng shard test; mỗi câu chỉ giữ top-k triple + ansRecall@k."""
    scorer.eval()
    scorer.rbf.mm = True                    # complete-path lúc test
    results = {}
    for rec in tqdm(stream.epoch(0), total=len(stream), desc='retrieve'):
        scores = score_record(scorer, rec, emb, device)
        top = torch.topk(scores, min(k, len(scores))).indices.numpy()
        names, rels = rec['entity_names'], rec['relation_list']
        h, r, t = rec['h'], rec['r'], rec['t']
        triples = [(names[h[i]], rels[r[i]], names[t[i]], float(scores[i])) for i in top]

        a_ent = set(rec['a_ent'])
        got = set(h[top].tolist()) | set(t[top].tolist())
        results[rec['id']] = {
            'question': rec['question'],
            'q_entity': [names[i] for i in rec['q_ent']],
            'a_entity_in_graph': [names[i] for i in rec['a_ent']],
            'scored_triples': triples,
        }
        pos = rec['label'] > 0                        # triples of the annotated answer subgraph
        for kk in (50, 100, 200):
            tk = top[:kk]
            got_k = set(h[tk].tolist()) | set(t[tk].tolist())
            results[rec['id']][f'ansR@{kk}'] = len(a_ent & got_k) / max(len(a_ent), 1)
            results[rec['id']][f'gtR@{kk}'] = float(pos[tk].sum() / max(pos.sum(), 1))
    return results


def mean_metric(results, key):
    return float(np.mean([v[key] for v in results.values()]))


emb_test = RelationOnlyEmb(OUT_DIR / f'lemb_{Z_ENCODER}_test')
test_stream = ShardStream(get_shards('test', 'test'), shuffle=False, chunk_shards=1, slim=False)
preds = retrieve_topk(scorer, test_stream, emb_test, DEVICE)
print(f'{len(preds):,} câu test | ' + '  '.join(f'{m}@{k} {mean_metric(preds, f"{m}@{k}"):.4f}' for m in ('ansR', 'gtR') for k in (50, 100, 200)))

json.dump(preds, open(OUT_DIR / 'runs' / RUN_NAME / 'test_preds.json', 'w'))
emb_test.close(); del emb_test, test_stream
free_memory()
report('SAU Phase 4')

## Before saving the notebook output
Removes the symlinks to `REUSE_DIR`, so the saved output holds only `runs/` and the new `prior.npy` files (not another copy of the shards).

In [ ]:
n = 0
for p in list(OUT_DIR.rglob('*')):
    if p.is_symlink():
        p.unlink(); n += 1
print(f'removed {n} symlinks; keep: {OUT_DIR / "runs" / RUN_NAME}')
report('cuối notebook')
